# DDP 資料分析
用 SQLite 載入原始資料，方便跨檔查詢與比對。

## 載入資料

In [ ]:
import sqlite3
import pandas as pd
import json
from pathlib import Path

BASE = Path(".")
DB   = BASE / "ddp.db"
conn = sqlite3.connect(DB)

def sql(query, rows=50):
    df = pd.read_sql(query, conn)
    print(f"{len(df):,} rows")
    display(df.head(rows))
    return df

# ── 1. User_Roles CSV（Zentera DDP Role → User 對應）─────────────────────
df_raw = pd.read_csv(BASE / "User_Roles_202603231446.csv", dtype=str).fillna("")
df_raw.columns = ["Customer", "Project", "Role", "Users", "Application", "AccessPolicies"]
df_raw.to_sql("roles", conn, if_exists="replace", index=False)

rows_ru = []
for _, r in df_raw.iterrows():
    for user in [u.strip() for u in r["Users"].split(",") if u.strip()]:
        rows_ru.append({"Role": r["Role"], "User": user.upper(), "Application": r["Application"]})
df_role_user = pd.DataFrame(rows_ru)
df_role_user.to_sql("role_user", conn, if_exists="replace", index=False)

# ── 2. Users CSV（Zentera 使用者：帳號、姓名、Role、最後登入、狀態）──────
df_users = pd.read_csv(BASE / "Users_202603231447.csv", dtype=str).fillna("")
df_users.columns = [c.strip() for c in df_users.columns]
df_users.rename(columns={
    "User(Account)":                                      "Account",
    "First Name":                                         "FirstName",  # 姓
    "Last Name":                                          "LastName",   # 名
    "Application(Server Group)/Service/MSG Device Group": "Application",
    "Last Access":                                        "LastAccessTs",
    "Last Access (UTC+08:00)":                            "LastAccess",
    "Expiration":                                         "ExpirationTs",
    "Expiration (UTC+08:00)":                             "Expiration",
    "Authentication Type":                                "AuthType",
}, inplace=True)
df_users["Account"] = df_users["Account"].str.upper()

# 修正：LastName 只有 1 字代表姓名填反了，對調回來
# e.g. FirstName=顯煜, LastName=邱 → FirstName=邱, LastName=顯煜
swapped = df_users["LastName"].str.len() == 1
df_users.loc[swapped, ["FirstName", "LastName"]] = (
    df_users.loc[swapped, ["LastName", "FirstName"]].values
)

df_users.to_sql("users", conn, if_exists="replace", index=False)

# ── 3. Server_Profiles CSV（Zentera VM Hostname → Role 對應）─────────────
df_servers = pd.read_csv(BASE / "Server_Profiles_202603231446.csv", dtype=str).fillna("")
df_servers.columns = [c.strip() for c in df_servers.columns]
df_servers.to_sql("servers", conn, if_exists="replace", index=False)

# ── 4. 工單 JSON ──────────────────────────────────────────────────────────
#df_tickets = pd.DataFrame(json.loads((BASE / "delta_tickets_clean.json").read_text(encoding="utf-8")))
#df_tickets.to_sql("tickets", conn, if_exists="replace", index=False)

# ── 5. AD 群組成員（groups_LTW_all.xlsx）──────────────────────────────────
xl = pd.ExcelFile(BASE / "groups_LTW_all.xlsx", engine="calamine")

df_groups = xl.parse("群組清單")
df_groups.columns = ["GroupName", "Description", "ManagedBy", "MemberCount", "BG"]
df_groups.to_sql("ad_groups", conn, if_exists="replace", index=False)

member_frames = []
for sheet in xl.sheet_names:
    if sheet == "群組清單":
        continue
    df = xl.parse(sheet)
    df.columns = ["GroupName", "ADAccount", "CN", "Mail", "BU"]
    df["BG"] = sheet
    df["ADAccount"] = df["ADAccount"].str.upper()
    member_frames.append(df)
df_members = pd.concat(member_frames, ignore_index=True)
df_members.to_sql("ad_members", conn, if_exists="replace", index=False)

# ── Index ──────────────────────────────────────────────────────────────────
for ddl in [
    "CREATE INDEX IF NOT EXISTS idx_role_user_user   ON role_user(User)",
    "CREATE INDEX IF NOT EXISTS idx_role_user_role   ON role_user(Role)",
    "CREATE INDEX IF NOT EXISTS idx_users_account    ON users(Account)",
    "CREATE INDEX IF NOT EXISTS idx_servers_hostname ON servers(Hostname)",
    "CREATE INDEX IF NOT EXISTS idx_ad_members_acct  ON ad_members(ADAccount)",
    "CREATE INDEX IF NOT EXISTS idx_ad_members_grp   ON ad_members(GroupName)",
]:
    conn.execute(ddl)
conn.commit()

print("\n已載入 SQLite ✓")
print(f"  roles      : {len(df_raw):,} rows")
print(f"  role_user  : {len(df_role_user):,} rows")
print(f"  users      : {len(df_users):,} rows  (對調修正：{swapped.sum():,} 筆)")
print(f"  servers    : {len(df_servers):,} rows")
#print(f"  tickets    : {len(df_tickets):,} rows")
print(f"  ad_groups  : {len(df_groups):,} rows")
print(f"  ad_members : {len(df_members):,} rows")


已載入 SQLite ✓
  roles      : 737 rows
  role_user  : 8,254 rows
  users      : 6,361 rows  (對調修正：2,172 筆)
  servers    : 804 rows
  tickets    : 15 rows
  ad_groups  : 1,500 rows
  ad_members : 54,664 rows


## 查詢區

### 各表欄位預覽（SELECT 範例）

In [34]:
# roles — Zentera Role 原始資料（一 Role 一行）
sql("SELECT Customer, Project, Role, Users, Application, AccessPolicies FROM roles LIMIT 5")

5 rows


,Customer,Project,Role,Users,Application,AccessPolicies
0,Delta,Onboarding flow project,ANSYS_Test,BEAR.HSU,,1
1,Delta,Onboarding flow project,BABG_CNWJ_LS LOB_EE,,,1
2,Delta,Onboarding flow project,BABG_CNWJ_LS LOB_ME,,,1
3,Delta,Onboarding flow project,BABG_TWPJ_BASBU_EE,JAY.LO,,1
4,Delta,Onboarding flow project,BABG_TWPJ_BASBU_FW,"JACOB.LEE,JIMMY.WP.SU,LUCENT.PENG,ODIN.HUANG",,1


,Customer,Project,Role,Users,Application,AccessPolicies
0,Delta,Onboarding flow project,ANSYS_Test,BEAR.HSU,,1
1,Delta,Onboarding flow project,BABG_CNWJ_LS LOB_EE,,,1
2,Delta,Onboarding flow project,BABG_CNWJ_LS LOB_ME,,,1
3,Delta,Onboarding flow project,BABG_TWPJ_BASBU_EE,JAY.LO,,1
4,Delta,Onboarding flow project,BABG_TWPJ_BASBU_FW,"JACOB.LEE,JIMMY.WP.SU,LUCENT.PENG,ODIN.HUANG",,1


In [35]:
# role_user — Role-User 展開（一 Role-User 一行）
sql("SELECT Role, User, Application FROM role_user LIMIT 5")

5 rows


,Role,User,Application
0,ANSYS_Test,BEAR.HSU,
1,BABG_TWPJ_BASBU_EE,JAY.LO,
2,BABG_TWPJ_BASBU_FW,JACOB.LEE,
3,BABG_TWPJ_BASBU_FW,JIMMY.WP.SU,
4,BABG_TWPJ_BASBU_FW,LUCENT.PENG,


,Role,User,Application
0,ANSYS_Test,BEAR.HSU,
1,BABG_TWPJ_BASBU_EE,JAY.LO,
2,BABG_TWPJ_BASBU_FW,JACOB.LEE,
3,BABG_TWPJ_BASBU_FW,JIMMY.WP.SU,
4,BABG_TWPJ_BASBU_FW,LUCENT.PENG,


In [36]:
# users — Zentera 使用者（帳號、姓名、最後登入、狀態）
sql("SELECT Account, FirstName, LastName, Role, Application, LastAccess, Status, AuthType FROM users LIMIT 5")

5 rows


,Account,FirstName,LastName,Role,Application,LastAccess,Status,AuthType
0,AARON.AC.CHIU,邱,顯煜,PSBG_TWTAO_DTD_LT,,2026-03-04 18:51:54,Active,External
1,AARON.CHUNG,鍾,仲修,ICTBG_TWPJ_AMSBD_FW,,2025-07-22 13:06:11,Active,External
2,AARON.CK.HUANG,黃,振國,"EVSBG_TWPJ_EMBU_ME,EVSBG_TWTAO_EMBU_ME",,2026-03-21 20:12:39,Active,External
3,AARON.HAUNG,黃,丞灝,"ICTBG_TWPJ_DCSBU_ME,ICTBG_TWTAO_DCSBU_ME",,2026-03-04 18:51:54,Active,External
4,AARON.HO,何,嘉倫,"ICTBG_TWPJ_IVNBD_EE,ICTBG_TWTAO_IVNBD_EE",,2026-03-04 18:51:54,Active,External


,Account,FirstName,LastName,Role,Application,LastAccess,Status,AuthType
0,AARON.AC.CHIU,邱,顯煜,PSBG_TWTAO_DTD_LT,,2026-03-04 18:51:54,Active,External
1,AARON.CHUNG,鍾,仲修,ICTBG_TWPJ_AMSBD_FW,,2025-07-22 13:06:11,Active,External
2,AARON.CK.HUANG,黃,振國,"EVSBG_TWPJ_EMBU_ME,EVSBG_TWTAO_EMBU_ME",,2026-03-21 20:12:39,Active,External
3,AARON.HAUNG,黃,丞灝,"ICTBG_TWPJ_DCSBU_ME,ICTBG_TWTAO_DCSBU_ME",,2026-03-04 18:51:54,Active,External
4,AARON.HO,何,嘉倫,"ICTBG_TWPJ_IVNBD_EE,ICTBG_TWTAO_IVNBD_EE",,2026-03-04 18:51:54,Active,External


In [37]:
# servers — Zentera VM Server Profiles（Hostname → Role 對應）
sql('SELECT Hostname, "App Profile", "Application(Server Group)", "Server Function", "Host IP", "Online Since (UTC+08:00)" FROM servers LIMIT 5')

5 rows


,Hostname,App Profile,Application(Server Group),Server Function,Host IP,Online Since (UTC+08:00)
0,THBPODDPDW01,Digital Design Platform,TRA_DET_UAT_Test,"Application Server,Access Server","fe80::5c62:bd8d:c585:89ea,10.150.206.10",2026-03-21 20:50:12
1,TWPJ1RDAECE01,Digital Design Platform,PSBG_TWPJ_PKG_CAETM,"Application Server,Access Server",10.143.184.170,2026-03-21 20:50:13
2,TWPJ1RDAECE02,Digital Design Platform,PSBG_TWPJ_PKG_CAETM,"Application Server,Access Server",10.143.184.171,2026-03-21 20:50:42
3,TWPJ1RDCE01,Digital Design Platform,PSBG_TWPJ_PKG_CAETM,"Application Server,Access Server",10.143.184.176,2026-03-21 20:50:24
4,TWPJ1RDCE02,Digital Design Platform,PSBG_TWPJ_PKG_CAETM,"Application Server,Access Server",10.143.184.177,2026-03-21 20:50:23


,Hostname,App Profile,Application(Server Group),Server Function,Host IP,Online Since (UTC+08:00)
0,THBPODDPDW01,Digital Design Platform,TRA_DET_UAT_Test,"Application Server,Access Server","fe80::5c62:bd8d:c585:89ea,10.150.206.10",2026-03-21 20:50:12
1,TWPJ1RDAECE01,Digital Design Platform,PSBG_TWPJ_PKG_CAETM,"Application Server,Access Server",10.143.184.170,2026-03-21 20:50:13
2,TWPJ1RDAECE02,Digital Design Platform,PSBG_TWPJ_PKG_CAETM,"Application Server,Access Server",10.143.184.171,2026-03-21 20:50:42
3,TWPJ1RDCE01,Digital Design Platform,PSBG_TWPJ_PKG_CAETM,"Application Server,Access Server",10.143.184.176,2026-03-21 20:50:24
4,TWPJ1RDCE02,Digital Design Platform,PSBG_TWPJ_PKG_CAETM,"Application Server,Access Server",10.143.184.177,2026-03-21 20:50:23


In [38]:
# tickets — IT 工單
sql("SELECT id, subject, requester, technician, status, created_time, site, category FROM tickets LIMIT 5")

5 rows


,id,subject,requester,technician,status,created_time,site,category
0,788824,新進同仁申請Digital Ｗorkplace_ROSA.CHENG鄭宜勳,ROSA.CHENG 鄭宜勳,JIAHUA.WU 吳家驊,Open,24/03/2026 10:11 AM,桃園二廠/Taoyuan Factory II,DDS
1,788823,新申請提供Nas folder 、NB Hostname、VM Hostname,ZANE.WU 吳文育,JIAHUA.WU 吳家驊,Open,24/03/2026 10:10 AM,桃園二廠/Taoyuan Factory II,DDS
2,787372,[DDP] 帳號加入VM系統,STACEY.WANG 王蓓瑩,JIAHUA.WU 吳家驊,Closed,20/03/2026 01:11 PM,中壢五廠/Chungli Factory V,IT DDP
3,786363,DDP相關問題,MARK.JY.LIN 林君穎,JIAHUA.WU 吳家驊,Closed,18/03/2026 07:03 PM,中壢五廠/Chungli Factory V,IT DDP
4,786139,DDP申請,BRAD.HUANG 黃柏睿,JIAHUA.WU 吳家驊,Closed,18/03/2026 03:32 PM,桃園二廠/Taoyuan Factory II,IT DDP


,id,subject,requester,technician,status,created_time,site,category
0,788824,新進同仁申請Digital Ｗorkplace_ROSA.CHENG鄭宜勳,ROSA.CHENG 鄭宜勳,JIAHUA.WU 吳家驊,Open,24/03/2026 10:11 AM,桃園二廠/Taoyuan Factory II,DDS
1,788823,新申請提供Nas folder 、NB Hostname、VM Hostname,ZANE.WU 吳文育,JIAHUA.WU 吳家驊,Open,24/03/2026 10:10 AM,桃園二廠/Taoyuan Factory II,DDS
2,787372,[DDP] 帳號加入VM系統,STACEY.WANG 王蓓瑩,JIAHUA.WU 吳家驊,Closed,20/03/2026 01:11 PM,中壢五廠/Chungli Factory V,IT DDP
3,786363,DDP相關問題,MARK.JY.LIN 林君穎,JIAHUA.WU 吳家驊,Closed,18/03/2026 07:03 PM,中壢五廠/Chungli Factory V,IT DDP
4,786139,DDP申請,BRAD.HUANG 黃柏睿,JIAHUA.WU 吳家驊,Closed,18/03/2026 03:32 PM,桃園二廠/Taoyuan Factory II,IT DDP


In [39]:
# ad_groups — AD 群組清單
sql("SELECT GroupName, Description, ManagedBy, MemberCount, BG FROM ad_groups LIMIT 5")

5 rows


,GroupName,Description,ManagedBy,MemberCount,BG
0,L-TW-1030G2,NaN,NaN,59,NaN
1,L-TW-1040G5,NaN,NaN,1,NaN
2,L-TW-1040G7,NaN,NaN,1,NaN
3,L-TW-4UD00MES,申請單號:2020020841800\n\n2020051911300,EVA.CE.CHEN 陳恒萱,14,"CORP, IT, SEA"
4,L-TW-ACABD-EE,Form ID 2023100151226\n\nForm ID 2023050261508,NICK.XIAO 蕭銘宏,19,"CORP, ICTBG"


,GroupName,Description,ManagedBy,MemberCount,BG
0,L-TW-1030G2,NaN,NaN,59,NaN
1,L-TW-1040G5,NaN,NaN,1,NaN
2,L-TW-1040G7,NaN,NaN,1,NaN
3,L-TW-4UD00MES,申請單號:2020020841800\n\n2020051911300,EVA.CE.CHEN 陳恒萱,14,"CORP, IT, SEA"
4,L-TW-ACABD-EE,Form ID 2023100151226\n\nForm ID 2023050261508,NICK.XIAO 蕭銘宏,19,"CORP, ICTBG"


In [40]:
# ad_members — AD 群組成員（BG 已合回）
sql("SELECT GroupName, ADAccount, CN, Mail, BU, BG FROM ad_members LIMIT 5")

5 rows


,GroupName,ADAccount,CN,Mail,BU,BG
0,L-TW-BlockToInternet,V-CHIAHUA,V-chiahua,None,-,-
1,L-TW-BlockToInternet,V-CYLIN,V-cylin,None,-,-
2,L-TW-BlockToInternet,V-SJ-HUANG,V-sj-huang,None,-,-
3,L-TW-BlockToInternet,V-TESTBOT018,V-testbot018,None,-,-
4,L-TW-BlockToInternet,V-TPPERNG,V-tpperng,None,-,-


,GroupName,ADAccount,CN,Mail,BU,BG
0,L-TW-BlockToInternet,V-CHIAHUA,V-chiahua,None,-,-
1,L-TW-BlockToInternet,V-CYLIN,V-cylin,None,-,-
2,L-TW-BlockToInternet,V-SJ-HUANG,V-sj-huang,None,-,-
3,L-TW-BlockToInternet,V-TESTBOT018,V-testbot018,None,-,-
4,L-TW-BlockToInternet,V-TPPERNG,V-tpperng,None,-,-


In [41]:
EXCLUDE_GROUPS = [
    'L-TW-AUTODESK-PUBLIC',
    'L-TW-BALSME01',
    'L-TW-DDC-PCAP-USER',
    'L-TW-DELTA-VDIAPP01',
    'L-TW-Deltabox3',
    'L-TW-GFTP',
    'L-TW-M365-STD',
    'L-TW-NolimitLocalAdminUser',
    'L-TW-PSO',
    'L-TW-SSLVPN',
    'L-TW-VDI-RDSPOOL-APP',
]

df_out = pd.read_sql("""
SELECT
    m.ADAccount                                         AS "AD Account",
    m.CN                                                AS "AD Name (Chinese Name)",
    u.FirstName,
    u.LastName,
    COALESCE(NULLIF(m.Mail, ''),
             m.ADAccount || '@deltaww.com')             AS "Mail",
    m.BG,
    m.BU,
    ''                                                  AS "Role",
    ''                                                  AS "NB Hostname",
    'G-Delta-rollout_admin'                             AS "Group Owner",
    m.GroupName                                         AS "Group Name",
    ''                                                  AS "NAS Folder Name",
    s.Hostname                                          AS "VM HostName",
    s."Host IP"                                         AS "Host IP",
    ''                                                  AS "NEW VM",
    ru.Role                                             AS "User Roles",
    u.Application                                       AS "Application",
    ''                                                  AS "Template Name",
    ''                                                  AS "Location"
FROM ad_members m
INNER JOIN users u     ON u.Account  = m.ADAccount
LEFT JOIN role_user ru ON ru.User    = m.ADAccount
LEFT JOIN servers s    ON s."Application(Server Group)" = ru.Role
ORDER BY m.BG, m.ADAccount, m.GroupName, s.Hostname
""", conn)

df_out = df_out[~df_out["Group Name"].isin(EXCLUDE_GROUPS)]

# Role = VM Hostname 數字前兩位英文（e.g. TWPJ1RDAECE01 → CE）
df_out["Role"] = df_out["VM HostName"].str.extract(r'([A-Za-z]{2})\d+$', expand=False)

out_path = "ad_user_vm_mapping.xlsx"
df_out.to_excel(out_path, index=False, sheet_name="AD User VM Mapping")
print(f"已輸出：{out_path}  ({len(df_out):,} rows)")

已輸出：ad_user_vm_mapping.xlsx  (89,032 rows)


In [42]:
# 查某人有哪些 DDP Role
sql("""
    SELECT Role, Application
    FROM role_user
    WHERE User = 'JIAHUA.WU'
    ORDER BY Role
""")

1 rows


,Role,Application
0,DDP-IT-ADMIN,


,Role,Application
0,DDP-IT-ADMIN,


In [43]:
# 查某人在哪些 AD 群組
sql("""
    SELECT GroupName, BG, BU
    FROM ad_members
    WHERE ADAccount = 'JIAHUA.WU'
    ORDER BY BG, GroupName
""")

1 rows


,GroupName,BG,BU
0,L-TW-SCP,IT,RDS&SI


,GroupName,BG,BU
0,L-TW-SCP,IT,RDS&SI


In [44]:
# 工單申請人 vs AD 群組
sql("""
    SELECT
        t.id,
        t.requester,
        t.status,
        GROUP_CONCAT(DISTINCT m.GroupName) AS ad_groups
    FROM tickets t
    LEFT JOIN ad_members m
        ON UPPER(TRIM(SUBSTR(t.requester, 1,
            CASE WHEN INSTR(t.requester, ' ') > 0
                 THEN INSTR(t.requester, ' ') - 1
                 ELSE LENGTH(t.requester) END
        ))) = m.ADAccount
    GROUP BY t.id, t.requester, t.status
    ORDER BY t.id DESC
    LIMIT 20
""")

15 rows


,id,requester,status,ad_groups
0,788824,ROSA.CHENG 鄭宜勳,Open,NaN
1,788823,ZANE.WU 吳文育,Open,NaN
2,787372,STACEY.WANG 王蓓瑩,Closed,"L-TW-PSBG-PACKAGE,L-TW-PSPKCE04,L-TW-USB-ENABL..."
3,786363,MARK.JY.LIN 林君穎,Closed,L-TW-PSNBDEE01
4,786139,BRAD.HUANG 黃柏睿,Closed,L-TW-FMPTSME18
5,786037,BRUCE.LAI 賴文評,Closed,L-TW-FMLCSME06
6,786029,IAN.LIAO 廖紘毅,Closed,L-TW-FMLCSME06
7,785432,DARREN.KR.LIN 林昆榮,Open,NaN
8,785290,CL.LIU 劉建隆,Closed,L-TW-PSAPFW02
9,783879,I-PECK.WU 吳培愷,Closed,L-TW-PSSPSEE02


,id,requester,status,ad_groups
0,788824,ROSA.CHENG 鄭宜勳,Open,NaN
1,788823,ZANE.WU 吳文育,Open,NaN
2,787372,STACEY.WANG 王蓓瑩,Closed,"L-TW-PSBG-PACKAGE,L-TW-PSPKCE04,L-TW-USB-ENABL..."
3,786363,MARK.JY.LIN 林君穎,Closed,L-TW-PSNBDEE01
4,786139,BRAD.HUANG 黃柏睿,Closed,L-TW-FMPTSME18
5,786037,BRUCE.LAI 賴文評,Closed,L-TW-FMLCSME06
6,786029,IAN.LIAO 廖紘毅,Closed,L-TW-FMLCSME06
7,785432,DARREN.KR.LIN 林昆榮,Open,NaN
8,785290,CL.LIU 劉建隆,Closed,L-TW-PSAPFW02
9,783879,I-PECK.WU 吳培愷,Closed,L-TW-PSSPSEE02
